# AskSight — QLoRA Fine-tuning: Qwen3-VL-4B-Instruct

Fine-tunes `Qwen/Qwen3-VL-4B-Instruct` using QLoRA (4-bit quantization + LoRA adapters)  
on **VizWiz-VQA**, **TextVQA**, and **DocVQA** for blind and low-vision assistive use.

**Run on:** Kaggle (free T4/P100 GPU, 30 hrs/week) or Google Colab (T4 GPU)

**What you get after training:**
- A `checkpoints/asksight-v1/` folder with LoRA adapter weights (~50 MB)
- Set `ASKSIGHT_MODEL_CHECKPOINT=checkpoints/asksight-v1` in backend `.env` to use it

**Estimated time:**
| Dataset size | Epochs | GPU | Time |
|---|---|---|---|
| 5 000 samples | 3 | Kaggle T4 | ~2 hrs |
| 15 000 samples | 3 | Kaggle T4 | ~6 hrs |
| 20 000 samples | 3 | Kaggle P100 | ~8 hrs |

## 0. Configuration

First attach these datasets in Kaggle via **Add Data**:
- [VizWiz-VQA](https://www.kaggle.com/datasets/lhanhsin/vizwiz) by `lhanhsin` → `/kaggle/input/vizwiz` (required)
- [Text VQA](https://www.kaggle.com/datasets/dmytruto/textvqa) by `dmytruto` → `/kaggle/input/textvqa` (recommended)
- [DocVQA](https://www.kaggle.com/datasets/suvroo/docvqa) by `suvroo` → `/kaggle/input/docvqa` (optional)

Edit these values before running. Everything else is automatic.

In [ ]:
import os
from pathlib import Path

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen3-VL-4B-Instruct"
OUTPUT_DIR      = os.environ.get(
    "ASKSIGHT_OUTPUT_DIR",
    "/kaggle/working/checkpoints/asksight-v1" if Path("/kaggle/working").is_dir() else "./checkpoints/asksight-v1",
)

# ── Dataset discovery ─────────────────────────────────────────────────────
# Leave the *_PATH overrides blank to auto-discover every annotation under DATA_DIR.
# An override may be a JSON/JSONL/CSV/TSV file or a dataset folder.
DATA_DIR        = Path(os.environ.get("ASKSIGHT_DATA_DIR", "/kaggle/input"))
VIZWIZ_PATH     = os.environ.get("VIZWIZ_PATH", "")
TEXTVQA_PATH    = os.environ.get("TEXTVQA_PATH", "")
DOCVQA_PATH     = os.environ.get("DOCVQA_PATH", "")

# ── QLoRA hyperparameters ─────────────────────────────────────────────────
LORA_RANK       = 16       # higher = more capacity, more VRAM
LORA_ALPHA      = 32       # usually 2x rank
LORA_DROPOUT    = 0.05

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS          = 3
BATCH_SIZE      = 2        # keep low — VLM inputs are large
GRAD_ACCUM      = 8        # effective batch = 2 × 8 = 16
LEARNING_RATE   = 2e-4
MAX_SAMPLES     = 7500     # Sized for Kaggle ~6-hour session (~5.5 hrs runtime)
SEED            = 42

## 1. Install dependencies

In [ ]:
!pip install -q \
    transformers>=4.51.0 \
    peft>=0.14.0 \
    bitsandbytes>=0.45.0 \
    accelerate>=1.3.0 \
    trl>=0.16.0 \
    qwen-vl-utils>=0.0.10 \
    pillow \
    opencv-python-headless

print("✅ Dependencies installed")

## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU detected. Switch to a GPU runtime."

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9
print(f"✅ GPU : {gpu.name}")
print(f"   VRAM: {vram_gb:.1f} GB")

if vram_gb < 14:
    print("⚠️  Less than 14 GB VRAM — reduce BATCH_SIZE to 1 if you get OOM errors.")

## 3. Load datasets

Recursively scans `DATA_DIR` (default `/kaggle/input`) for annotation files and images.
TextVQA and VizWiz use separate parsers because their filename fields differ.
Each file is inspected and reported by split: annotation count, image-field match rate,
missing images/questions/answers, and loaded samples.

Records with answers go into `samples` for training. Test/inference records without
ground-truth answers go into `inference_samples` with `answer=""`; no answers are fabricated.
Training is blocked if train/validation annotations have missing images or questions.

In [ ]:
import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

TAGS = ("textvqa", "vizwiz", "docvqa")
SPLITS = ("train", "val", "test")
DATASET_LINKS = {
    "textvqa": "https://www.kaggle.com/datasets/dmytruto/textvqa",
    "vizwiz": "https://www.kaggle.com/datasets/lhanhsin/vizwiz",
    "docvqa": "https://www.kaggle.com/datasets/suvroo/docvqa",
}
ANNOTATION_SUFFIXES = {".json", ".jsonl", ".csv", ".tsv"}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def get_value(record, *names):
    wanted = {name.lower() for name in names}
    for key, value in record.items():
        if str(key).lower() in wanted and value not in (None, "", [], {}):
            return value
    return None


def find_field(record, *names):
    wanted = {name.lower() for name in names}
    for key, value in record.items():
        if str(key).lower() in wanted and value not in (None, "", [], {}):
            return str(key), value
    return None, None


def as_text(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        for key in ("text", "question", "answer", "value", "label"):
            text = as_text(value.get(key))
            if text:
                return text
    return ""


def answer_text(value):
    if isinstance(value, (list, tuple)):
        texts = [answer_text(v) for v in value]
        texts = [t for t in texts if t]
        return Counter(texts).most_common(1)[0][0] if texts else ""
    return as_text(value)


def unanswerable_flag(record):
    lowered = {str(k).lower(): v for k, v in record.items()}
    if "unanswerable" in lowered:
        return str(lowered["unanswerable"]).lower() in {"1", "true", "yes"}
    if "answerable" in lowered:
        return str(lowered["answerable"]).lower() in {"0", "false", "no"}
    return False


def read_annotations(path):
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        with path.open(encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    if suffix in (".csv", ".tsv"):
        with path.open(encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter="\t" if suffix == ".tsv" else ",")
            return list(reader)
    content = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(content, dict):
        for key in ("data", "annotations", "questions", "items", "records", "samples"):
            if isinstance(content.get(key), list):
                return content[key]
        return [content]
    return content if isinstance(content, list) else []


def flatten(rows):
    out = []
    for row in rows:
        nested = get_value(row, "questions", "annotations", "data", "items") if isinstance(row, dict) else None
        if isinstance(nested, list) and nested and all(isinstance(x, dict) for x in nested):
            out.extend(nested)
        else:
            out.append(row)
    return out


def files_in(value):
    path = Path(value)
    if path.is_file() and path.suffix.lower() in ANNOTATION_SUFFIXES:
        return [path]
    if path.is_dir():
        return sorted(p for p in path.rglob("*") if p.is_file() and p.suffix.lower() in ANNOTATION_SUFFIXES)
    return []


def detect_tag(path, rows):
    hint = str(path).lower() + " " + json.dumps(rows[:3], ensure_ascii=False)[:4000].lower()
    for tag in TAGS:
        if tag in hint:
            return tag
    sample = next((r for r in rows if isinstance(r, dict)), {})
    answers = get_value(sample, "answers", "answer", "responses", "gt_answer")
    if "unanswerable" in sample or "answerable" in sample:
        return "vizwiz"
    if isinstance(answers, list) and answers:
        if isinstance(answers[0], dict):
            answer_keys = {str(k).lower() for k in answers[0]}
            if "text" in answer_keys:
                return "docvqa"
            if "answer" in answer_keys or "confidence" in answer_keys:
                return "textvqa"
        elif isinstance(answers[0], str):
            return "vizwiz"
    return "unknown"


def detect_split(path):
    text = str(path).lower()
    for split in SPLITS:
        if re.search(rf"(?:^|[^a-z]){split}(?:[^a-z]|$)", text):
            return split
    return "unknown"


def parse_textvqa(row):
    image_field, image_value = find_field(
        row, "image_id", "imageId", "image", "image_path", "filename", "file_name", "path"
    )
    return {
        "question": as_text(get_value(row, "question", "question_text", "query")),
        "answer": answer_text(get_value(row, "answers", "answer", "gt_answer", "response")),
        "unanswerable": unanswerable_flag(row),
        "image_field": image_field,
        "image_value": image_value,
    }


def parse_vizwiz(row):
    image_field, image_value = find_field(
        row, "image", "image_path", "image_name", "filename", "file_name", "image_id", "imageId"
    )
    answer = answer_text(get_value(row, "answers", "answer", "gt_answer"))
    unanswerable = unanswerable_flag(row)
    if answer.lower() in {"unanswerable", "cannot answer", "no answer"}:
        unanswerable = True
    if unanswerable:
        answer = ""
    return {
        "question": as_text(get_value(row, "question", "question_text", "query")),
        "answer": answer,
        "unanswerable": unanswerable,
        "image_field": image_field,
        "image_value": image_value,
    }


def parse_generic(row):
    image_field, image_value = find_field(
        row, "image_path", "image_file", "image_name", "image", "filename", "file_name",
        "img_path", "path", "image_id", "imageId", "imgId"
    )
    return {
        "question": as_text(get_value(row, "question", "question_text", "query", "prompt", "text")),
        "answer": answer_text(get_value(row, "answer", "answers", "response", "gt_answer", "label", "ground_truth")),
        "unanswerable": unanswerable_flag(row),
        "image_field": image_field,
        "image_value": image_value,
    }


PARSERS = {"textvqa": parse_textvqa, "vizwiz": parse_vizwiz}
DEFAULT_PARSER = parse_generic

print(f"Data dir: {DATA_DIR}")

dataset_dirs = []
image_files = []
annotation_files = []
if DATA_DIR.is_dir():
    dataset_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
    image_files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    annotation_files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in ANNOTATION_SUFFIXES)

image_index = defaultdict(list)
for image in image_files:
    for key in (image.name, image.stem):
        image_index[key].append(image)
    match = re.search(r"(?:^|[_-])(\d+)$", image.stem)
    if match:
        digits = match.group(1)
        image_index[digits].append(image)
        image_index[str(int(digits))].append(image)


def resolve_image(value, annotation_path):
    if value is None:
        return None
    if isinstance(value, dict):
        value = get_value(value, "path", "file_name", "file", "name", "image", "id")
    text = str(value).strip()
    if not text:
        return None
    p = Path(text)
    if annotation_path.is_relative_to(DATA_DIR):
        rel = annotation_path.relative_to(DATA_DIR)
        root = DATA_DIR / rel.parts[0] if len(rel.parts) > 1 else DATA_DIR
    else:
        root = annotation_path.parent
    candidates = [p, DATA_DIR / p, root / p, annotation_path.parent / p,
                  root / p.name, annotation_path.parent / p.name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    for key in (p.name, p.stem, text, text.lstrip("0") or text):
        for candidate in image_index.get(key, []):
            if candidate.is_file():
                return candidate
    return None


rows_by_file = {}
skipped_files = Counter()
selected = []
for tag, value in (("textvqa", TEXTVQA_PATH), ("vizwiz", VIZWIZ_PATH), ("docvqa", DOCVQA_PATH)):
    for path in files_in(value) if value else []:
        selected.append((path, tag, detect_split(path)))
covered = {path for path, _, _ in selected}
for path in annotation_files:
    if path in covered or path in rows_by_file:
        continue
    try:
        rows_by_file[path] = flatten(read_annotations(path))
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        skipped_files[f"{type(exc).__name__}: {exc}"[:100]] += 1
        continue
    if rows_by_file[path]:
        selected.append((path, detect_tag(path, rows_by_file[path]), detect_split(path)))
for path, _, _ in selected:
    if path in rows_by_file:
        continue
    try:
        rows_by_file[path] = flatten(read_annotations(path))
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        skipped_files[f"{type(exc).__name__}: {exc}"[:100]] += 1

print(f"Dataset dirs ({len(dataset_dirs)}):")
for path in dataset_dirs:
    print(f"  {path}")
print(f"Annotation files found: {len(selected)}")
for path, tag, split in selected:
    print(f"  {tag:10s} {split:8s} {path}  ({len(rows_by_file.get(path, []))} annotations)")
print(f"Image files found: {len(image_files)}")
image_dirs = Counter(p.parent for p in image_files)
print(f"Image dirs ({len(image_dirs)}):")
for path, count in sorted(image_dirs.items()):
    print(f"  {path}  ({count} images)")
print("Image filename examples:")
for path in image_files[:5]:
    print(f"  {path.name}")
for reason, count in skipped_files.items():
    print(f"  ⚠️  skipped annotation file ({count}): {reason}")

for path, tag, split in selected:
    rows = rows_by_file.get(path, [])
    print(f"\n=== Raw inspection: {tag}/{split} {path.name} ===")
    print(f"  structure: {type(rows).__name__} with {len(rows)} records")
    if rows:
        sample_row = rows[0] if isinstance(rows[0], dict) else {}
        print(f"  keys: {sorted(sample_row) if sample_row else type(rows[0]).__name__}")
        field_counts = Counter()
        for row in rows:
            if isinstance(row, dict):
                field, _ = find_field(
                    row, "image_id", "imageId", "image", "image_path", "image_name",
                    "filename", "file_name", "path", "img_path"
                )
                field_counts[field or "none"] += 1
        print(f"  image fields used: {dict(field_counts)}")
        field, value = find_field(
            sample_row, "image_id", "imageId", "image", "image_path", "image_name",
            "filename", "file_name", "path", "img_path"
        )
        print(f"  image field example: {field}={value!r}")
        print(f"  answer key in first record: {get_value(sample_row, 'answers', 'answer', 'gt_answer') is not None}")
        for i, row in enumerate(rows[:4], 1):
            print(f"  raw[{i}]: {str(row)[:600]}")

samples = []
inference_samples = []
reports = []

for path, tag, split in selected:
    rows = rows_by_file.get(path, [])
    parser = PARSERS.get(tag, DEFAULT_PARSER)
    stats = Counter()
    skip_reasons = Counter()
    examples = defaultdict(list)
    for row in rows:
        stats["total"] += 1
        if not isinstance(row, dict):
            skip_reasons["record is not an object"] += 1
            continue
        parsed = parser(row)
        question = parsed["question"]
        answer = parsed["answer"]
        unanswerable = parsed["unanswerable"]
        image_value = parsed["image_value"]

        if not question:
            stats["missing_questions"] += 1
            skip_reasons["missing question"] += 1
            if len(examples["missing question"]) < 5:
                examples["missing question"].append(str(row)[:200])
            continue
        if image_value is None:
            stats["missing_images"] += 1
            skip_reasons["no image field"] += 1
            if len(examples["no image field"]) < 5:
                examples["no image field"].append(str(row)[:200])
            continue
        image = resolve_image(image_value, path)
        if image is None:
            stats["missing_images"] += 1
            skip_reasons["image file not found"] += 1
            if len(examples["image file not found"]) < 5:
                examples["image file not found"].append(str(image_value))
            continue
        stats["matched_images"] += 1

        if not answer and not unanswerable:
            stats["missing_answers"] += 1

        record = {
            "image_path": str(image),
            "question": question,
            "answer": answer,
            "source": tag,
            "split": split,
            "unanswerable": unanswerable,
        }
        if split == "test":
            inference_samples.append(record)
            stats["inference_samples"] += 1
        elif answer or unanswerable:
            samples.append(record)
            stats["trainable_samples"] += 1
        else:
            inference_samples.append(record)
            stats["inference_samples"] += 1

    stats["loaded_samples"] = stats["trainable_samples"] + stats["inference_samples"]
    report = {
        "tag": tag, "split": split, "path": path,
        "total": stats["total"],
        "matched_images": stats["matched_images"],
        "missing_images": stats["missing_images"],
        "missing_questions": stats["missing_questions"],
        "missing_answers": stats["missing_answers"],
        "loaded_samples": stats["loaded_samples"],
        "trainable_samples": stats["trainable_samples"],
        "inference_samples": stats["inference_samples"],
        "skip_reasons": dict(skip_reasons),
        "examples": dict(examples),
    }
    reports.append(report)

for report in reports:
    print(f"\n--- {report['tag']}/{report['split']} {report['path'].name} ---")
    print(f"  total annotations     : {report['total']:,}")
    print(f"  matched images        : {report['matched_images']:,}")
    print(f"  missing images        : {report['missing_images']:,}")
    print(f"  missing questions     : {report['missing_questions']:,}")
    print(f"  missing answers       : {report['missing_answers']:,}")
    print(f"  successfully loaded   : {report['loaded_samples']:,}")
    print(f"    trainable           : {report['trainable_samples']:,}")
    print(f"    inference-only      : {report['inference_samples']:,}")
    print(f"  skip reasons          : {report['skip_reasons'] or 'none'}")
    for reason, values in report["examples"].items():
        print(f"    {reason} examples: {values}")

print("\n=== Dataset loading report ===")
print(f"{'dataset':10s} {'split':8s} {'total':>8s} {'matched':>8s} {'no_img':>7s} "
      f"{'no_q':>6s} {'no_ans':>7s} {'loaded':>8s} {'train':>7s} {'infer':>7s}")
for report in reports:
    print(f"{report['tag']:10s} {report['split']:8s} {report['total']:8,d} "
          f"{report['matched_images']:8,d} {report['missing_images']:7,d} "
          f"{report['missing_questions']:6,d} {report['missing_answers']:7,d} "
          f"{report['loaded_samples']:8,d} {report['trainable_samples']:7,d} "
          f"{report['inference_samples']:7,d}")

blocking = []
for report in reports:
    if report["tag"] in ("textvqa", "vizwiz") and report["split"] in ("train", "val"):
        if report["missing_images"]:
            blocking.append(f"{report['tag']}/{report['split']}: {report['missing_images']} missing images")
        if report["missing_questions"]:
            blocking.append(f"{report['tag']}/{report['split']}: {report['missing_questions']} missing questions")
        if report["trainable_samples"] == 0:
            blocking.append(f"{report['tag']}/{report['split']}: no trainable samples")

assert not blocking, "❌ Dataset validation failed:\n  " + "\n  ".join(blocking)
assert samples, "❌ No trainable samples with answers — cannot start QLoRA training."
assert all(Path(s["image_path"]).is_file() for s in samples), "❌ A trainable sample has a missing image file."
assert all(s["question"] and isinstance(s["answer"], str) and s["source"] for s in samples), (
    "❌ Normalized trainable samples are missing question, answer, or source."
)
print(f"\n✅ Dataset validation passed — {len(samples):,} trainable, {len(inference_samples):,} inference-only records.")

if MAX_SAMPLES and len(samples) > MAX_SAMPLES:
    import random
    random.seed(SEED)
    random.shuffle(samples)
    samples = samples[:MAX_SAMPLES]
    print(f"  → Capped to {MAX_SAMPLES:,} trainable samples (MAX_SAMPLES)")

source_counts = Counter(s["source"] for s in samples)
print(f"\n✅ Total trainable: {len(samples):,} samples")
print("   Mix  :", dict(source_counts))

## 4. Load model + processor (4-bit QLoRA)

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading processor from {MODEL_ID} ...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
# Set pixel bounds on image_processor directly (avoids FutureWarning in transformers v5+)
processor.image_processor.min_pixels = 256 * 32 * 32
processor.image_processor.max_pixels = 1024 * 32 * 32
print("✅ Processor loaded")

print(f"Loading model in 4-bit from {MODEL_ID} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,   # extra memory saving
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("✅ Model loaded")

# Freeze vision encoder — only language decoder LoRA layers train
if hasattr(model, "visual") and model.visual is not None:
    for param in model.visual.parameters():
        param.requires_grad = False
    print("✅ Vision encoder frozen")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~10M trainable / 4B total (~0.25%)

## 5. Dataset class

In [ ]:
import re
from torch.utils.data import Dataset
from PIL import Image

DESCRIBE_PROMPT = (
    "You are a visual assistant for a person with low vision. "
    "Describe this image in 2 to 4 short, clear spoken sentences. "
    "Prioritize in this exact order: (1) Immediate physical hazards or obstacles, "
    "(2) Main objects or people and their positions using clock-face directions, "
    "(3) Important readable signs or labels. "
    "Output plain spoken text only. Keep your response under 60 words."
)

QUESTION_PROMPT = (
    "You are a visual assistant for a person with low vision. "
    "Answer the user's question directly and concisely based strictly on the image. "
    "Read any requested numbers, dosages, expiry dates, or labels word-for-word. "
    "If the requested text or object is blurry, obscured, or cut off, "
    "do not guess — tell the user clearly how to adjust the camera. "
    "Output plain spoken text only and keep your answer under 40 words."
)

UNANSWERABLE_ANSWER = (
    "The image is unclear or cut off. Please adjust the camera angle and try again."
)

def clean_tts_text(text: str) -> str:
    """Strip Markdown so the answer is TTS-ready."""
    text = re.sub(r"#+\s*", "", text)
    text = re.sub(r"[\*_]+", "", text)
    text = re.sub(r"^\s*[-*•]\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"`+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


class AssistiveDataset(Dataset):
    """Multimodal dataset for VizWiz, TextVQA, and DocVQA."""

    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        # Load image
        image_path = item.get("image_path")
        if image_path and Path(image_path).exists():
            image = Image.open(image_path).convert("RGB")
        else:
            image = Image.new("RGB", (336, 336), color=(128, 128, 128))

        question = item.get("question", "").strip()
        answer   = item.get("answer", "").strip()

        # Unanswerable examples (VizWiz) → teach safe refusal
        if item.get("unanswerable") and not answer:
            answer = UNANSWERABLE_ANSWER

        # Dual-mode routing
        is_describe = not question or question.lower() in ["describe", "describe this photo."]
        system_prompt = DESCRIBE_PROMPT if is_describe else QUESTION_PROMPT
        user_text     = "Describe this photo." if is_describe else question

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": user_text},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": clean_tts_text(answer)}]},
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False)
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True,
        )

        sample = {k: v[0] for k, v in inputs.items()}
        sample["labels"] = sample["input_ids"].clone()
        return sample


train_dataset = AssistiveDataset(samples, processor)
print(f"✅ Dataset ready — {len(train_dataset):,} samples")

# Quick sanity check on one sample
sample = train_dataset[0]
print(f"   input_ids shape : {sample['input_ids'].shape}")
print(f"   labels shape    : {sample['labels'].shape}")

## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    bf16                        = True,
    logging_steps               = 10,
    save_strategy               = "epoch",
    save_total_limit            = 2,          # keep last 2 checkpoints only
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,      # required for multimodal inputs
    report_to                   = "none",     # set "wandb" if you have it
    seed                        = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
)

print(f"🚀 Starting training — {EPOCHS} epoch(s), {len(train_dataset):,} samples")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Steps per epoch    : {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print()

trainer.train()

## 7. Save checkpoint

In [ ]:
import json
from datetime import datetime

# Save LoRA adapter weights + processor config
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Save training metadata alongside the checkpoint
meta = {
    "model_id"       : MODEL_ID,
    "trained_at"     : datetime.utcnow().isoformat() + "Z",
    "epochs"         : EPOCHS,
    "total_samples"  : len(samples),
    "source_mix"     : dict(Counter(s["source"] for s in samples)),
    "lora_rank"      : LORA_RANK,
    "lora_alpha"     : LORA_ALPHA,
    "learning_rate"  : LEARNING_RATE,
    "batch_size"     : BATCH_SIZE,
    "grad_accum"     : GRAD_ACCUM,
}
with open(f"{OUTPUT_DIR}/training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Checkpoint saved to: {OUTPUT_DIR}")
print(f"   Files:")
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"     {p.name:40s}  {size_mb:6.1f} MB")

## 8. Test inference with the fine-tuned model
Quick smoke test — does the checkpoint answer correctly?

In [ ]:
from PIL import Image as PILImage

# Smoke-test with a real image from the loaded dataset — no internet needed.
record = next(s for s in samples if Path(s["image_path"]).is_file())
test_image = PILImage.open(record["image_path"]).convert("RGB")
TEST_QUESTION = record["question"] or "Describe this image."

model.eval()
with torch.no_grad():
    messages = [
        {"role": "system", "content": QUESTION_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": TEST_QUESTION},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[test_image], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        top_p=0.8,
        do_sample=True,
    )
    answer = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)

print(f"Question : {TEST_QUESTION}")
print(f"Answer   : {clean_tts_text(answer)}")

## 9. Use in AskSight backend

After downloading the checkpoint from Kaggle, point the backend at it:

```bash
# In backend/.env
ASKSIGHT_MODEL_CHECKPOINT="/absolute/path/to/checkpoints/asksight-v1"
ASK_TIMEOUT_SECONDS=30   # fine-tuned on GPU is fast — 30s is enough
```

The backend's `inference.py` auto-detects that the checkpoint directory contains  
`adapter_config.json` and loads it as a LoRA adapter on top of the base model.

```python
# This happens automatically in qwen3_vl_engine.py:
if (checkpoint / "adapter_config.json").is_file():
    base_model_id = PeftConfig.from_pretrained(checkpoint).base_model_name_or_path
    # loads Qwen3-VL-4B-Instruct + your LoRA on top
```

No code changes needed in the backend — just update `.env` and restart.

## 10. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment and run if you want to push to HuggingFace Hub

# from huggingface_hub import notebook_login
# notebook_login()

# HF_REPO = "your-username/asksight-qwen3-vl-4b-qlora"
# model.push_to_hub(HF_REPO, private=True)
# processor.push_to_hub(HF_REPO, private=True)
# print(f"✅ Pushed to https://huggingface.co/{HF_REPO}")